# CIFAR-10: attention-head faults in the output projection W_o

The crash fault of the original study zeroes a head's output before the heads
are concatenated and projected by `W_o`. Because

`W_o(concat(head_0..head_H)) = sum_h W_o[:, h*d_k:(h+1)*d_k] @ head_h + b_o`,

zeroing head *h*'s `d_k` columns of `W_o` removes exactly the same term. A crash
is therefore one particular **weight fault in W_o**. This notebook corrupts
those columns in other ways and compares the outcome with the crash.

It loads the checkpoint trained by `notebooks/tiny_gpt2/cifar10.ipynb`
(no training here) and uses the same fixed evaluation subset. Results go to
`results/tiny_gpt2/cifar10/wo_faults/`; the original results are not
touched. Method and outputs: `docs/wo_faults.md`.

| Step | Question |
|---|---|
| E0 | Does zeroing W_o columns reproduce the crash exactly? (sanity) |
| E1 | Single head, every head: how harmful are bit flips and stuck-at values compared with the crash, and do they hit the same heads? |
| E2 | Random multi-head faults on the same head sets as the crash study |
| D1 | Most critical heads: damage vs the **value** a weight is changed to |
| D2 | Most critical heads: damage vs the **original magnitude** of the changed weight |
| D3 | Most critical heads: damage vs the **float32 bit** that is flipped |
| D4 | Pooled: is damage explained by the size of the change alone? |


## Setup (imports, globals, data split and model copied from the original notebook)

In [1]:
import json, pathlib, sys, time

import numpy as np
import pandas as pd
import torch

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from faulttol import config as C
from faulttol import multihead, plots, saferegion, single_head
from faulttol.injection import collect_attention_stats_per_head, get_all_heads
from faulttol.runtime import configure_threads, device_report, select_device
from faulttol.training import make_loader

print("torch", torch.__version__, "| threads", configure_threads(),
      "| cuda build", torch.version.cuda, "| cuda available", torch.cuda.is_available())
print("root:", ROOT)

torch 2.14.0+cu126 | threads 8 | cuda build 12.6 | cuda available True
root: D:\Laboratory\fault-tolerance-models-and-task-comparison


In [2]:
RESULTS_DIR = ROOT / "results" / "tiny_gpt2" / "cifar10"
MODEL_NAME = "tiny_gpt2_cifar10"
ARCHIVE = ROOT / "data" / "cifar-10-python.tar.gz"
# NVIDIA GPU when one is visible; FT_DEVICE=cpu reproduces the CPU-only run
DEVICE = select_device()

IMAGE_SIZE, PATCH_SIZE, NUM_CLASSES = 32, 4, 10
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
MEAN = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32)
STD = np.array([0.2470, 0.2435, 0.2616], dtype=np.float32)

MODEL_CONFIG = dict(C.TINY_GPT2_PARAM, activation="gelu")

TRAIN_CONFIG = {
    "loss": "CrossEntropyLoss",
    "optimizer": "AdamW",
    # a from-scratch ViT on 32x32 images needs a larger step and stronger decay
    # than the EXP_A synthetic task; cosine annealing follows the ViT recipe of
    # the Dropout-and-Outliers paper, which uses this same corpus
    "learning_rate": 6e-4,
    "weight_decay": 5e-2,
    "batch_size": 128,
    "epochs": C._env_int("FT_EPOCHS", 40),
    "patience": C._env_int("FT_PATIENCE", 8),
    "grad_clip": 1.0,
    "label_smoothing": 0.1,
    "selection_metric": "roc_auc",
    "cosine_schedule": True,
    "augmentation": "RandomCrop(32, padding=4) + RandomHorizontalFlip",
}

CRIT_COL = "D_pos_mean"
DEG_COL = "D_mean"
SAFE_COL = "D_pos_mean"
DEG_LABEL = "mean probability damage"
SAFE_LABEL = "mean positive damage"
TAUS = C.CLS_TAUS
TAU_LABELS = ["tau_%.3f" % t for t in TAUS]

print("device:", device_report(DEVICE))
print("patches %dx%d = %d tokens | epochs %d"
      % (IMAGE_SIZE // PATCH_SIZE, IMAGE_SIZE // PATCH_SIZE,
         (IMAGE_SIZE // PATCH_SIZE) ** 2, TRAIN_CONFIG["epochs"]))

device: cuda:0 NVIDIA GeForce RTX 4070 Laptop GPU | 8.0 GB | sm_89 | torch cuda 12.6 | tf32 off
patches 8x8 = 64 tokens | epochs 40


In [3]:
import pickle, tarfile

def load_cifar10():
    train_batches, test_batch = [], None
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        for member in tar.getmembers():
            name = pathlib.PurePosixPath(member.name).name
            if name.startswith("data_batch_"):
                train_batches.append((name, pickle.load(tar.extractfile(member), encoding="bytes")))
            elif name == "test_batch":
                test_batch = pickle.load(tar.extractfile(member), encoding="bytes")
    train_batches.sort(key=lambda kv: kv[0])

    def to_arrays(batch):
        x = batch[b"data"].reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
        y = np.asarray(batch[b"labels"], dtype=np.int64)
        return (x - MEAN[None, :, None, None]) / STD[None, :, None, None], y

    xs, ys = zip(*[to_arrays(b) for _, b in train_batches])
    tx, ty = to_arrays(test_batch)
    return np.concatenate(xs), np.concatenate(ys), tx, ty

train_x, train_y, test_x, test_y = load_cifar10()
print("raw: train=%s test=%s" % (train_x.shape, test_x.shape))
print("class balance:", np.bincount(train_y))

raw: train=(50000, 3, 32, 32) test=(10000, 3, 32, 32)
class balance: [5000 5000 5000 5000 5000 5000 5000 5000 5000 5000]


In [4]:
import torch.nn.functional as F

def augment_batch(xb):
    """RandomCrop(32, padding=4) + RandomHorizontalFlip, batched.

    Vectorised via advanced indexing: with the model on the GPU a per-sample
    Python loop here is the slowest part of the epoch.  The random draws are
    taken in the same order and shape as the loop version, so the augmentation
    stream is unchanged.
    """
    b = xb.size(0)
    padded = F.pad(xb, (4, 4, 4, 4), mode="reflect")
    ox = torch.randint(0, 9, (b,))
    oy = torch.randint(0, 9, (b,))
    flip = torch.rand(b) < 0.5

    ar = torch.arange(32)
    rows = (oy.view(b, 1) + ar.view(1, 32)).view(b, 1, 32, 1)
    # a horizontal flip is the crop read with its columns reversed
    cols = torch.where(flip.view(b, 1), (31 - ar).view(1, 32), ar.view(1, 32))
    cols = (ox.view(b, 1) + cols).view(b, 1, 1, 32)

    return padded[torch.arange(b).view(b, 1, 1, 1),
                  torch.arange(3).view(1, 3, 1, 1),
                  rows, cols]

def balanced_subset(y, n_total, seed):
    rng = np.random.default_rng(seed)
    classes = np.unique(y)
    per = n_total // len(classes)
    picks = [rng.choice(np.flatnonzero(y == c), size=per, replace=False) for c in classes]
    out = np.concatenate(picks)
    rng.shuffle(out)
    return out

rng = np.random.default_rng(C.SPLIT_SEED)
perm = rng.permutation(len(train_x))
n_val = 5_000
val_idx, tr_idx = perm[:n_val], perm[n_val:]

n_sub = C._env_int("FT_TRAIN_SUBSAMPLE", 0)
if n_sub:
    tr_idx = tr_idx[:n_sub]

X_train, y_train = torch.from_numpy(train_x[tr_idx]), train_y[tr_idx]
X_val, y_val = torch.from_numpy(train_x[val_idx]), train_y[val_idx]

X_test_all = torch.from_numpy(test_x)
eval_idx = balanced_subset(test_y, C.N_FAULT, C.FAULT_SPLIT_SEED)
calib_mask = np.ones(len(test_y), dtype=bool)
calib_mask[eval_idx] = False
X_calib, y_calib = X_test_all[calib_mask], test_y[calib_mask]
X_eval, y_eval = X_test_all[eval_idx].contiguous(), test_y[eval_idx]

print("train=%d val=%d calib=%d fault_eval=%d"
      % (len(X_train), len(X_val), len(X_calib), len(X_eval)))
print("fault-eval class balance:", np.bincount(y_eval))

train=45000 val=5000 calib=9000 fault_eval=1000
fault-eval class balance: [100 100 100 100 100 100 100 100 100 100]


In [5]:
from faulttol.metrics import (classification_clean_metrics, classification_damage,
                              classification_outputs)
from faulttol.model import ImageClassifier, count_params
from faulttol.training import train_classifier

torch.manual_seed(C.TRAIN_SEED)
np.random.seed(C.TRAIN_SEED)

model = ImageClassifier(MODEL_CONFIG, image_size=IMAGE_SIZE, patch_size=PATCH_SIZE,
                        in_channels=3, num_classes=NUM_CLASSES, activation="gelu")
print("params = %d" % count_params(model))
print("tokens = %d patches (+CLS)" % model.n_patches)
print("heads  = %d" % len(get_all_heads(model)))
model

params = 10669834
tokens = 64 patches (+CLS)
heads  = 36


ImageClassifier(
  (pos): LearnedPositionalEmbedding()
  (layers): ModuleList(
    (0-5): 6 x EncoderLayer(
      (attn): MultiHeadAttention(
        (W_q): Linear(in_features=384, out_features=384, bias=False)
        (W_k): Linear(in_features=384, out_features=384, bias=False)
        (W_v): Linear(in_features=384, out_features=384, bias=False)
        (W_o): Linear(in_features=384, out_features=384, bias=False)
      )
      (ff): Sequential(
        (0): Linear(in_features=384, out_features=1536, bias=False)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.05, inplace=False)
        (3): Linear(in_features=1536, out_features=384, bias=False)
      )
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=False)
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=False)
      (dropout): Dropout(p=0.05, inplace=False)
    )
  )
  (norm_out): LayerNorm((384,), eps=1e-05, elementwise_affine=True, bias=False)
  (dropout): Dropout(p=0

In [6]:
from faulttol import wo_faults as W
from faulttol.metrics import (classification_clean_metrics, classification_damage,
                              classification_outputs)
from faulttol.training import load_checkpoint
from scipy.stats import kendalltau, pearsonr, spearmanr

EVAL_CHUNK = 500
OUT = RESULTS_DIR / "wo_faults"
TABLES, FIGURES = OUT / "tables", OUT / "figures"
for d in (TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
CKPT = RESULTS_DIR / "models" / "best.pt"
SHIPPED = json.loads((RESULTS_DIR / "summary.json").read_text(encoding="utf-8"))

CRIT_COL = "D_pos_mean"   # criticality / damage signal, as in the crash study
DEG_COL = "D_mean"        # degradation variable for the severity analysis
SAFE_COL = "D_pos_mean"   # safe-region degradation variable
TAUS = C.CLS_TAUS
TAU_LABELS = ["tau_%.3f" % t for t in TAUS]

def save(df, name):
    path = TABLES / name
    df.to_csv(path, index=False)
    print("saved", path.relative_to(ROOT), df.shape)
    return path

print("checkpoint:", CKPT.relative_to(ROOT), "| exists:", CKPT.is_file())
print("artifacts ->", OUT.relative_to(ROOT))
print("N_FAULT=%d  WO_REPEATS=%d  WO_K_TRIALS=%d  WO_SAMPLES=%d"
      % (C.N_FAULT, C.WO_REPEATS, C.WO_K_TRIALS, C.WO_DEEPDIVE_SAMPLES))


checkpoint: results\tiny_gpt2\cifar10\models\best.pt | exists: True
artifacts -> results\tiny_gpt2\cifar10\wo_faults
N_FAULT=1000  WO_REPEATS=10  WO_K_TRIALS=2000  WO_SAMPLES=30


## Load the trained checkpoint

In [7]:
ckpt = load_checkpoint(CKPT, model)
model.to(DEVICE).eval()
CHECKSUM = W.model_checksum(model)
print("loaded epoch %d, best val score %.4f" % (ckpt["epoch"], ckpt["best_score"]))

def evaluate():
    return classification_outputs(model, X_eval, y_eval, DEVICE, chunk=EVAL_CHUNK)

damage_fn = classification_damage
clean_eval = classification_clean_metrics(evaluate())
print("clean (fixed eval subset, n=%d):" % len(X_eval), clean_eval)
if C.N_FAULT == SHIPPED["n_fault_eval_samples"]:
    for k, v in clean_eval.items():
        if abs(v - SHIPPED["clean_metrics"][k]) > 1e-9:
            print("WARNING: clean %s %.6f differs from summary.json %.6f - "
                  "is the checkpoint from the same run?" % (k, v, SHIPPED["clean_metrics"][k]))
    print("clean metrics checked against summary.json")

t0 = time.time()
for _ in range(5):
    evaluate()
per_eval = (time.time() - t0) / 5.0
print("%.3f s per evaluation" % per_eval)


loaded epoch 28, best val score 0.9673


clean (fixed eval subset, n=1000): {'accuracy': 0.758, 'precision': 0.7568988922097417, 'recall': 0.758, 'f1': 0.7561815694260791, 'roc_auc': 0.9710555555555554}
clean metrics checked against summary.json


0.300 s per evaluation


## Crash reference

The single-head crash analysis is re-run on the loaded checkpoint (32
evaluations). It gives the reference criticality, the severity weights `W` and
the critical-head set used by the multi-head study, and picks the deep-dive
heads: the top 3 by crash criticality plus the least critical head.


In [8]:
crash_df = single_head.run_single_head_analysis(
    model=model, evaluate=evaluate, damage_fn=damage_fn,
    criticality_col=CRIT_COL, model_name=MODEL_NAME, verbose=False)
save(crash_df, "crash_single_head.csv")

shipped_sh = pd.read_csv(RESULTS_DIR / "tables" / "single_head_damage.csv")
if C.N_FAULT == SHIPPED["n_fault_eval_samples"]:
    diff = np.abs(shipped_sh["criticality"].to_numpy() - crash_df["criticality"].to_numpy()).max()
    print("max |criticality - shipped| = %.2e" % diff,
          "(OK)" if diff < 1e-6 else "<- WARNING: differs from the shipped crash run")

weight_lookup = single_head.build_head_weight_lookup(crash_df)
critical_heads = single_head.critical_head_set(crash_df, C.CRITICAL_TOP_FRACTION)
DEEP_HEADS = W.select_deepdive_heads(crash_df, C.WO_DEEPDIVE_N_CRITICAL)
print("critical heads:", ["L%d_H%d" % h for h in critical_heads])
print("deep-dive heads:", ["L%d_H%d (%s)" % h for h in DEEP_HEADS])
M_LAYER = W.layer_wo_absmax(model)
print("max|W_o| per layer (m):", ["%.4f" % m for m in M_LAYER])
crash_df.nlargest(5, "criticality")[["head_name", "criticality", DEG_COL]]


saved results\tiny_gpt2\cifar10\wo_faults\tables\crash_single_head.csv (36, 30)
max |criticality - shipped| = 9.89e-17 (OK)
critical heads: ['L0_H5', 'L0_H3', 'L0_H2', 'L0_H0']
deep-dive heads: ['L0_H5 (top1)', 'L0_H3 (top2)', 'L0_H2 (top3)', 'L4_H0 (least)']
max|W_o| per layer (m): ['0.2124', '0.2678', '0.2199', '0.2456', '0.1871', '0.1722']


,head_name,criticality,D_mean
5,L0_H5,0.380327,0.361777
3,L0_H3,0.130469,0.091595
2,L0_H2,0.075919,0.042604
0,L0_H0,0.072276,0.047797
4,L0_H4,0.053319,0.027695


## E0: sanity check

Zeroing a head's `W_o` columns must give **bit-identical** outputs to the crash
mask, for every single head and for 50 random multi-head sets (the first 50
configurations of the crash study). Two more checks exercise the bit-flip code:
a bit error rate of 0, and the same bit flipped twice, must both equal the clean
model.


In [9]:
sanity = W.run_wo_sanity(model, evaluate, n_multi=C.WO_SANITY_MULTI,
                         seed=C.MULTIHEAD_FAULT_SEED, fault_seed=C.WO_FAULT_SEED)
save(sanity, "sanity_checks.csv")
sanity_summary = sanity.groupby("check").agg(n=("exact", "size"), all_exact=("exact", "all"),
                                             max_abs_diff=("max_abs_diff", "max"))
assert sanity["exact"].all(), "W_o fault does not reproduce the expected model state"
assert W.model_checksum(model) == CHECKSUM, "weights were not restored"
print("all sanity checks exact; weights restored")
sanity_summary


saved results\tiny_gpt2\cifar10\wo_faults\tables\sanity_checks.csv (158, 4)
all sanity checks exact; weights restored


,n,all_exact,max_abs_diff
check,,,
ber0_vs_clean,36,True,0.0
double_flip_vs_clean,36,True,0.0
zero_vs_mask_multi,50,True,0.0
zero_vs_mask_single,36,True,0.0


## E1: single-head W_o faults, every head

Each head in turn gets one W_o fault while all other heads stay intact:

- **bit flips** at bit error rate (BER) 1e-6 ... 1e-3: every bit of the head's
  `W_o` columns flips independently with probability BER;
- **stuck-at**: a random fraction of the head's `W_o` weights is stuck at 0,
  at +m or at -m, where m = max|W_o| of the head's layer.

Stochastic levels are repeated `WO_REPEATS` times. The W_o criticality of a head
is the mean `D_pos_mean` over repeats, the same formula as the crash
criticality. `catastrophic_rate` is the share of trials whose outputs became
non-finite (counted as failed predictions, see `docs/wo_faults.md`).


In [10]:
fault_grid = W.build_fault_grid(C.WO_BER_LEVELS, C.WO_STUCK_VALUES_REL, C.WO_STUCK_FRACS)
wo_sh = W.run_wo_single_head(model, evaluate, damage_fn, fault_grid, C.WO_REPEATS,
                             seed=C.WO_FAULT_SEED, model_name=MODEL_NAME)
save(wo_sh, "wo_single_head.csv")
assert W.model_checksum(model) == CHECKSUM

wo_crit = W.wo_criticality(wo_sh, CRIT_COL)
save(wo_crit, "wo_single_head_criticality.csv")
crash_cmp = W.compare_with_crash(wo_crit, crash_df)
save(crash_cmp, "wo_vs_crash_single_head.csv")

plots.wo_level_curves(wo_crit, crash_df, "%s - single-head W_o fault criticality" % MODEL_NAME,
                      FIGURES / "wo_single_head_levels.png", ylabel=CRIT_COL)
plots.crash_vs_wo_criticality(wo_crit, crash_df, "%s - crash vs W_o criticality" % MODEL_NAME,
                              FIGURES / "crash_vs_wo_criticality.png")
crash_cmp


    single-head 500/4788  3.2 evals/s  eta 22.2 min


    single-head 1000/4788  3.2 evals/s  eta 19.4 min


    single-head 1500/4788  3.2 evals/s  eta 16.9 min


    single-head 2000/4788  3.2 evals/s  eta 14.3 min


    single-head 2500/4788  3.2 evals/s  eta 11.8 min


    single-head 3000/4788  3.2 evals/s  eta 9.2 min


    single-head 3500/4788  3.2 evals/s  eta 6.6 min


    single-head 4000/4788  3.2 evals/s  eta 4.1 min


    single-head 4500/4788  3.2 evals/s  eta 1.5 min


saved results\tiny_gpt2\cifar10\wo_faults\tables\wo_single_head.csv (4788, 38)
saved results\tiny_gpt2\cifar10\wo_faults\tables\wo_single_head_criticality.csv (576, 13)
saved results\tiny_gpt2\cifar10\wo_faults\tables\wo_vs_crash_single_head.csv (16, 9)


,fault_model,level,mean_criticality,max_criticality,most_critical_head,ratio_to_crash_mean,spearman_vs_crash,top3_overlap_with_crash,catastrophic_rate
0,bitflip,ber=1e-06,0.012225,0.054940,L4_H0,0.317288,0.142857,0,0.011111
1,bitflip,ber=1e-05,0.122140,0.274727,L3_H0,3.170011,0.114286,0,0.080556
2,bitflip,ber=0.0001,0.515828,0.549375,L4_H3,13.387744,0.041086,0,0.605556
3,bitflip,ber=0.001,0.549371,0.549371,L0_H0,14.258317,NaN,1,1.000000
4,stuck_at_0,frac=0.01,0.000417,0.002020,L0_H5,0.010818,0.868468,3,0.000000
5,stuck_at_0,frac=0.1,0.002654,0.014065,L0_H5,0.068888,0.982497,3,0.000000
6,stuck_at_0,frac=0.5,0.015187,0.116615,L0_H5,0.394168,0.996654,2,0.000000
7,stuck_at_0,frac=1,0.038530,0.380327,L0_H5,1.000000,1.000000,3,0.000000
8,stuck_at_+m,frac=0.01,0.001818,0.008172,L0_H5,0.047182,0.721493,3,0.000000
9,stuck_at_+m,frac=0.1,0.006596,0.033630,L0_H5,0.171204,0.797683,3,0.000000


## E2: random multi-head W_o faults

The head sets are drawn with the crash study's seed, so trial *k* here faults
exactly the heads that crash trial *k* crashed; only the fault differs. Each
failed head gets the W_o fault of the setting. The analysis is the crash
study's: degradation grid, rank correlations of severity vs degradation (plus
the size of the weight change `rel_change` and the number of flipped bits), and
the 95% safe regions.


In [11]:
def severity_correlations(df, deg_col, severities):
    rows = []
    for sev in severities:
        x, y = df[sev].to_numpy(float), df[deg_col].to_numpy(float)
        fin = np.isfinite(x) & np.isfinite(y)
        if np.std(x[fin]) == 0:
            continue
        pr = pearsonr(x[fin], y[fin]).statistic
        sr, sp = spearmanr(x, y)        # ranks: inf is just the largest value
        kr = kendalltau(x, y).statistic
        rows.append({"severity": sev, "degradation": deg_col, "pearson_finite": pr,
                     "spearman": sr, "kendall": kr, "spearman_p": sp})
    return pd.DataFrame(rows)

n_heads = len(get_all_heads(model))
crash_mh = pd.read_csv(RESULTS_DIR / "tables" / "random_multihead.csv")
mh_frames, mh_summary, curves = {}, [], {}
if C.N_FAULT == SHIPPED["n_fault_eval_samples"]:
    k = min(C.WO_K_TRIALS, len(crash_mh))
    curves["crash (shipped, same head sets)"] = (crash_mh["N_fail"][:k], crash_mh[DEG_COL][:k])

for setting, spec in C.WO_MULTIHEAD_SETTINGS:
    print("setting", setting)
    df = W.run_wo_multihead(model, evaluate, damage_fn, spec, setting, weight_lookup,
                            critical_heads, MODEL_NAME, C.WO_K_TRIALS,
                            seed=C.MULTIHEAD_FAULT_SEED, fault_seed=C.WO_FAULT_SEED,
                            num_layers=model.num_layers)
    assert W.model_checksum(model) == CHECKSUM
    mh_frames[setting] = df
    save(df, "wo_multihead_%s.csv" % setting)

    grid = multihead.degradation_grid(df, DEG_COL)
    save(grid.reset_index(), "wo_degradation_grid_%s.csv" % setting)
    plots.degradation_grid_plot(grid, "%s - W_o %s: mean %s" % (MODEL_NAME, setting, DEG_COL),
                                FIGURES / ("wo_degradation_grid_%s.png" % setting),
                                value_label=DEG_COL)
    corr = severity_correlations(df, DEG_COL, ("N_fail", "L_fault", "N_crit_fail", "W_fail",
                                               "W_fail_mean", "rel_change", "n_bitflips",
                                               "n_exp_msb_flips"))
    save(corr, "wo_severity_correlations_%s.csv" % setting)
    grids, safe = saferegion.run_safe_region_analysis(
        df, SAFE_COL, TAUS, n_heads, len(critical_heads),
        min_configs=C.MIN_CONFIGS_PER_BOUNDARY, safe_prob=C.SAFE_PROB, tau_labels=TAU_LABELS)
    save(grids, "wo_safe_region_grid_%s.csv" % setting)
    save(safe, "wo_safe_region_summary_%s.csv" % setting)
    curves["W_o " + setting] = (df["N_fail"], df[DEG_COL])

    row = {"setting": setting, "k_trials": len(df),
           "mean_" + DEG_COL: float(df[DEG_COL].mean()),
           "catastrophic_trial_rate": float((df["nonfinite_rate"] > 0).mean()),
           "mean_nonfinite_sample_rate": float(df["nonfinite_rate"].mean()),
           "spearman_N_fail": float(corr.set_index("severity").loc["N_fail", "spearman"]),
           "safe_regions": safe.to_dict(orient="records")}
    if C.N_FAULT == SHIPPED["n_fault_eval_samples"]:
        paired = crash_mh[DEG_COL].to_numpy()[:len(df)]
        row["crash_mean_" + DEG_COL + "_same_sets"] = float(paired.mean())
        row["share_trials_worse_than_crash"] = float(np.mean(df[DEG_COL].to_numpy() > paired))
    mh_summary.append(row)

plots.degradation_curves(curves, "%s - degradation vs failed heads: crash vs W_o faults" % MODEL_NAME,
                         FIGURES / "wo_multihead_curves.png", ylabel="mean " + DEG_COL)
pd.DataFrame(mh_summary).drop(columns="safe_regions")


setting bitflip_ber1e-4


    bitflip_ber1e-4 500/2000  3.3 evals/s  eta 7.5 min


    bitflip_ber1e-4 1000/2000  3.3 evals/s  eta 5.0 min


    bitflip_ber1e-4 1500/2000  3.3 evals/s  eta 2.5 min


    bitflip_ber1e-4 2000/2000  3.3 evals/s  eta 0.0 min


saved results\tiny_gpt2\cifar10\wo_faults\tables\wo_multihead_bitflip_ber1e-4.csv (2000, 47)
saved results\tiny_gpt2\cifar10\wo_faults\tables\wo_degradation_grid_bitflip_ber1e-4.csv (5, 5)


... (4 outputs left out)

    stuck_+m_frac0.10 2000/2000  3.0 evals/s  eta 0.0 min


saved results\tiny_gpt2\cifar10\wo_faults\tables\wo_multihead_stuck_+m_frac0.10.csv (2000, 47)
saved results\tiny_gpt2\cifar10\wo_faults\tables\wo_degradation_grid_stuck_+m_frac0.10.csv (5, 5)


saved results\tiny_gpt2\cifar10\wo_faults\tables\wo_severity_correlations_stuck_+m_frac0.10.csv (6, 6)
saved results\tiny_gpt2\cifar10\wo_faults\tables\wo_safe_region_grid_stuck_+m_frac0.10.csv (525, 9)
saved results\tiny_gpt2\cifar10\wo_faults\tables\wo_safe_region_summary_stuck_+m_frac0.10.csv (3, 9)


,setting,k_trials,mean_D_mean,catastrophic_trial_rate,mean_nonfinite_sample_rate,spearman_N_fail,crash_mean_D_mean_same_sets,share_trials_worse_than_crash
0,bitflip_ber1e-4,2000,0.543251,0.8515,0.849466,0.041207,0.359782,0.8925
1,stuck_+m_frac0.10,2000,0.034169,0.0000,0.000000,0.924652,0.359782,0.0000


## Deep-dive on the most critical heads

The heads are the top 3 by crash criticality plus the least critical head as a
baseline. Every trial changes the `W_o` columns of **one** head and nothing
else. `m` is max|W_o| of the head's layer.


### D1: damage vs the value a weight is changed to

Each value in the grid is written to:

- **single**: 50 random single weights, one at a time;
- **row**: 8 random whole rows (the `d_k` weights that write one output feature);
- **column**: 8 random whole columns (the `d_model` weights that read one of the
  head's `d_k` output dimensions);
- **block**: all of the head's `W_o` weights.

The same targets are reused for every value, so values are compared on paired
targets. Expect the column and block curves to be **flat**: a constant column
adds the same number to every output feature, and every LayerNorm that later
reads it subtracts it again, so any constant there acts exactly like zero.


In [12]:
d1 = W.run_value_sweep(model, evaluate, damage_fn, DEEP_HEADS, C.WO_VALUE_GRID,
                       n_single=C.WO_SWEEP_SINGLE, n_columns=C.WO_SWEEP_COLUMNS,
                       n_rows=C.WO_SWEEP_ROWS, seed=C.WO_FAULT_SEED, model_name=MODEL_NAME)
assert W.model_checksum(model) == CHECKSUM
save(d1, "deep_value_sweep.csv")
d1_table = d1.pivot_table(index=["granularity", "value_label"], columns="head_name",
                          values=CRIT_COL, aggfunc="mean", sort=False)
save(d1_table.reset_index(), "deep_value_sweep_summary.csv")
plots.value_sweep_plot(d1, CRIT_COL, "%s - D1: damage vs the value W_o weights are set to"
                       % MODEL_NAME, FIGURES / "deep_value_sweep.png")
d1_table


    value sweep 500/3484  3.2 evals/s  eta 15.6 min


    value sweep 1000/3484  3.2 evals/s  eta 12.9 min


    value sweep 1500/3484  3.2 evals/s  eta 10.3 min


    value sweep 2000/3484  3.2 evals/s  eta 7.8 min


    value sweep 2500/3484  3.2 evals/s  eta 5.1 min


    value sweep 3000/3484  3.2 evals/s  eta 2.5 min


saved results\tiny_gpt2\cifar10\wo_faults\tables\deep_value_sweep.csv (3484, 42)
saved results\tiny_gpt2\cifar10\wo_faults\tables\deep_value_sweep_summary.csv (52, 6)


head_name                   L0_H5     L0_H3     L0_H2     L4_H0
granularity value_label                                        
single      0            0.000100  0.000049  0.000042  0.000005
            +m/2         0.000199  0.000143  0.000098  0.000018
            -m/2         0.000212  0.000143  0.000118  0.000015
            +m           0.000393  0.000286  0.000209  0.000034
            -m           0.000410  0.000287  0.000226  0.000032
            +10m         0.004195  0.002987  0.002285  0.000339
            -10m         0.004275  0.003049  0.002244  0.000330
            +1e2         0.367201  0.341265  0.244349  0.039064
            -1e2         0.364615  0.339963  0.244438  0.038559
            +1e4         0.549407  0.548820  0.548445  0.518772
            -1e4         0.549383  0.549572  0.548205  0.518626
            +1e6         0.550523  0.550020  0.550494  0.550048
            -1e6         0.550663  0.550855  0.550325  0.549777
row         0            0.001942  0.001073  0.000753  0.000084
            +m/2         0.003174  0.001977  0.001146  0.000085
            -m/2         0.002709  0.001783  0.001273  0.000123
            +m           0.005342  0.003572  0.002050  0.000127
            -m           0.004711  0.003327  0.002149  0.000178
            +10m         0.092857  0.060964  0.029254  0.001362
            -10m         0.091089  0.056784  0.029271  0.001390
            +1e2         0.536049  0.542771  0.534829  0.203624
            -1e2         0.542943  0.536895  0.533233  0.205013
            +1e4         0.545584  0.555023  0.552113  0.539699
            -1e4         0.554571  0.546643  0.549231  0.542957
            +1e6         0.545604  0.555135  0.552407  0.548905
            -1e6         0.554660  0.546625  0.549326  0.551872
column      0            0.006387  0.002380  0.001872  0.000319
            +m/2         0.006387  0.002380  0.001872  0.000319
            -m/2         0.006387  0.002380  0.001872  0.000319
            +m           0.006387  0.002380  0.001872  0.000319
            -m           0.006387  0.002380  0.001872  0.000319
            +10m         0.006387  0.002380  0.001872  0.000319
            -10m         0.006387  0.002380  0.001872  0.000319
            +1e2         0.006387  0.002380  0.001872  0.000319
            -1e2         0.006387  0.002380  0.001872  0.000319
            +1e4         0.006389  0.002381  0.001876  0.000320
            -1e4         0.006390  0.002381  0.001876  0.000320
            +1e6         0.014866  0.016459  0.014352  0.003431
            -1e6         0.014882  0.016582  0.014278  0.003439
block       0            0.380327  0.130469  0.075919  0.011821
            +m/2         0.380327  0.130469  0.075919  0.011821
            -m/2         0.380327  0.130469  0.075919  0.011821
            +m           0.380327  0.130469  0.075919  0.011821
            -m           0.380327  0.130469  0.075919  0.011821
            +10m         0.380327  0.130469  0.075919  0.011821
            -10m         0.380327  0.130469  0.075919  0.011821
            +1e2         0.380327  0.130470  0.075919  0.011821
            -1e2         0.380327  0.130470  0.075918  0.011821
            +1e4         0.380371  0.130350  0.075959  0.011831
            -1e4         0.380314  0.130393  0.075943  0.011831
            +1e6         0.388129  0.351846  0.334264  0.029605
            -1e6         0.387258  0.347436  0.336888  0.029525

### D2: damage vs the original magnitude of the changed weight

The head's weights are ranked by |w| into deciles (`d01` smallest ... `d10`
largest) plus the top 1% and top 0.1%. From each bin, single weights are
**zeroed**, **negated**, **multiplied by 10** or get their **exponent MSB
flipped**, one weight per trial. Finally each whole bin is zeroed at once: is
removing the 10% largest weights worse than removing the 10% smallest?


In [13]:
d2 = W.run_magnitude_study(model, evaluate, damage_fn, DEEP_HEADS, C.WO_DEEPDIVE_SAMPLES,
                           seed=C.WO_FAULT_SEED, model_name=MODEL_NAME)
assert W.model_checksum(model) == CHECKSUM
save(d2, "deep_magnitude.csv")
d2_table = d2.pivot_table(index=["granularity", "op", "magnitude_bin"], columns="head_name",
                          values=CRIT_COL, aggfunc="mean", sort=False)
save(d2_table.reset_index(), "deep_magnitude_summary.csv")
plots.magnitude_bins_plot(d2, CRIT_COL, "%s - D2: damage vs |w| of the changed weight"
                          % MODEL_NAME, FIGURES / "deep_magnitude.png")

# rank correlation between |w| and damage, per head and operation (single weights).
# NaN means the damage was constant, e.g. every exponent-MSB flip is catastrophic.
import warnings
d2_single = d2[d2["granularity"] == "single"].copy()
d2_single["abs_old"] = d2_single["old_value"].abs()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    d2_corr = (d2_single.groupby(["head_name", "role", "op"])
               .apply(lambda g: spearmanr(g["abs_old"], g[CRIT_COL]).statistic,
                      include_groups=False)
               .rename("spearman_abs_w_vs_damage").reset_index())
save(d2_corr, "deep_magnitude_correlation.csv")
d2_corr.pivot(index=["head_name", "role"], columns="op", values="spearman_abs_w_vs_damage")


    magnitude 500/5728  3.3 evals/s  eta 26.8 min


    magnitude 1000/5728  3.3 evals/s  eta 24.2 min


    magnitude 1500/5728  3.3 evals/s  eta 21.6 min


    magnitude 2000/5728  3.3 evals/s  eta 19.0 min


    magnitude 2500/5728  3.3 evals/s  eta 16.5 min


    magnitude 3000/5728  3.3 evals/s  eta 13.9 min


... (4 outputs left out)

    magnitude 5500/5728  3.3 evals/s  eta 1.1 min


saved results\tiny_gpt2\cifar10\wo_faults\tables\deep_magnitude.csv (5728, 44)
saved results\tiny_gpt2\cifar10\wo_faults\tables\deep_magnitude_summary.csv (60, 7)


saved results\tiny_gpt2\cifar10\wo_faults\tables\deep_magnitude_correlation.csv (16, 4)


,op,flip_exp_msb,negate,x10,zero
head_name,role,,,,
L0_H2,top3,0.117344,0.958179,0.960867,0.958180
L0_H3,top2,0.038206,0.961062,0.960917,0.961085
L0_H5,top1,-0.174062,0.961683,0.960743,0.961655
L4_H0,least,0.076357,0.969069,0.970648,0.969041


### D3: damage vs the float32 bit that is flipped

For every bit position 0..31 (0-22 mantissa, 23-30 exponent with 30 the MSB, 31
sign) the same random weights get that one bit flipped, one weight per trial.


In [14]:
d3 = W.run_bit_position_study(model, evaluate, damage_fn, DEEP_HEADS, C.WO_DEEPDIVE_SAMPLES,
                              seed=C.WO_FAULT_SEED, model_name=MODEL_NAME)
assert W.model_checksum(model) == CHECKSUM
save(d3, "deep_bit_position.csv")
d3_table = d3.groupby(["head_name", "role", "bit"]).agg(
    damage_mean=(CRIT_COL, "mean"), damage_max=(CRIT_COL, "max"),
    catastrophic_rate=("nonfinite_rate", lambda s: float(np.mean(s > 0)))).reset_index()
save(d3_table, "deep_bit_position_summary.csv")
plots.bit_position_plot(d3, CRIT_COL, "%s - D3: damage per flipped bit of a W_o weight"
                        % MODEL_NAME, FIGURES / "deep_bit_position.png")
d3.groupby(["bit_field", "role"]).agg(
    damage_mean=(CRIT_COL, "mean"),
    catastrophic_rate=("nonfinite_rate", lambda s: float(np.mean(s > 0)))).unstack()


    bit position 500/3840  3.6 evals/s  eta 15.6 min


    bit position 1000/3840  3.6 evals/s  eta 13.2 min


    bit position 1500/3840  3.6 evals/s  eta 10.9 min


    bit position 2000/3840  3.6 evals/s  eta 8.5 min


    bit position 2500/3840  3.6 evals/s  eta 6.2 min


    bit position 3000/3840  3.6 evals/s  eta 3.9 min


    bit position 3500/3840  3.6 evals/s  eta 1.6 min


saved results\tiny_gpt2\cifar10\wo_faults\tables\deep_bit_position.csv (3840, 41)
saved results\tiny_gpt2\cifar10\wo_faults\tables\deep_bit_position_summary.csv (128, 6)


damage_mean                               catastrophic_rate  \
role              least      top1      top2      top3             least   
bit_field                                                                 
exponent   6.868966e-02  0.068919  0.068807  0.068799          0.041667   
mantissa   2.170167e-07  0.000002  0.000001  0.000001          0.000000   
sign       1.267529e-05  0.000145  0.000093  0.000071          0.000000   

                                         
role           top1      top2      top3  
bit_field                                
exponent   0.041667  0.054167  0.033333  
mantissa   0.000000  0.000000  0.000000  
sign       0.000000  0.000000  0.000000

### D4: is damage explained by the size of the change?

All deep-dive trials pooled, as damage against `||dW||_F`, the Frobenius norm
of the change to the head's `W_o` columns. If the curves of the critical heads
lie above the least critical head at the same `||dW||`, *where* the change
lands matters, not only how large it is.


In [15]:
pooled = W.pooled_delta_table({"value_sweep": d1, "magnitude": d2, "bit_position": d3}, CRIT_COL)
binned = W.binned_damage_by_delta(pooled, CRIT_COL)
save(binned, "deep_damage_vs_delta.csv")
plots.delta_damage_plot(pooled, binned, CRIT_COL,
                        "%s - D4: damage vs size of the W_o change" % MODEL_NAME,
                        FIGURES / "deep_damage_vs_delta.png")
fin = pooled[np.isfinite(pooled["delta_norm"])]
d4_corr = fin.groupby("role").apply(
    lambda g: spearmanr(g["delta_norm"], g[CRIT_COL]).statistic, include_groups=False)
print("Spearman(||dW||, damage) by head role (finite changes):")
print(d4_corr.round(3).to_string())


saved results\tiny_gpt2\cifar10\wo_faults\tables\deep_damage_vs_delta.csv (34, 7)


Spearman(||dW||, damage) by head role (finite changes):
role
least    0.993
top1     0.988
top2     0.988
top3     0.988


## Summary

In [16]:
def _top_findings():
    by_field = d3.groupby("bit_field").agg(damage_mean=(CRIT_COL, "mean"),
        catastrophic_rate=("nonfinite_rate", lambda s: float(np.mean(s > 0))))
    return {
        "bit_position_by_field": by_field.to_dict(orient="index"),
        "damage_vs_delta_spearman_by_role": {k: float(v) for k, v in d4_corr.items()},
        "magnitude_spearman": d2_corr.to_dict(orient="records"),
    }

summary = {
    "model_name": MODEL_NAME,
    "checkpoint": str(CKPT.relative_to(ROOT)),
    "fault_location": "W_o.weight[:, h*d_k:(h+1)*d_k] (the head's columns of the output projection)",
    "n_fault_eval_samples": int(C.N_FAULT),
    "clean_metrics": clean_eval,
    "seconds_per_evaluation": per_eval,
    "parameters": {
        "wo_repeats": C.WO_REPEATS, "wo_k_trials": C.WO_K_TRIALS,
        "ber_levels": list(C.WO_BER_LEVELS), "stuck_values_rel": list(C.WO_STUCK_VALUES_REL),
        "stuck_fracs": list(C.WO_STUCK_FRACS),
        "multihead_settings": [{"setting": s, "spec": sp} for s, sp in C.WO_MULTIHEAD_SETTINGS],
        "value_grid": [list(v) for v in C.WO_VALUE_GRID],
        "sweep_single": C.WO_SWEEP_SINGLE, "sweep_rows": C.WO_SWEEP_ROWS,
        "sweep_columns": C.WO_SWEEP_COLUMNS,
        "deepdive_samples": C.WO_DEEPDIVE_SAMPLES, "fault_seed": C.WO_FAULT_SEED,
        "m_layer": M_LAYER,
    },
    "deep_dive_heads": [{"head": "L%d_H%d" % (l, h), "role": r,
                         "crash_criticality": float(crash_df.loc[(crash_df["layer"] == l)
                                                                 & (crash_df["head"] == h),
                                                                 "criticality"].iloc[0])}
                        for l, h, r in DEEP_HEADS],
    "sanity": sanity_summary.reset_index().to_dict(orient="records"),
    "single_head_vs_crash": crash_cmp.to_dict(orient="records"),
    "multihead": mh_summary,
    **_top_findings(),
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
assert W.model_checksum(model) == CHECKSUM
print("wrote", (OUT / "summary.json").relative_to(ROOT))


wrote results\tiny_gpt2\cifar10\wo_faults\summary.json
